# CSE 151B — SFT data prep

Prepare external math datasets into a unified training schema for QLoRA SFT. See `docs/sft/pipeline.md`.

**Sources implemented:** MATH train (`EleutherAI/hendrycks_math`), **NuminaMath-CoT** (`AI-MO/NuminaMath-CoT`), **AGIEval/GaoKao MCQ** (`hails/agieval-*`), **OpenR1-Math-220k** (`open-r1/OpenR1-Math-220k`, sft-002a primary). Remaining: s1K-1.1 fallback, self-distilled format-fix.

Each source writes:

```text
data/sft_sources/<source>_ready.jsonl
data/sft_sources/<source>_stats.json
data/sft_sources/<source>_rejects.jsonl
```

Run sections in order. Do not mix until every source passes validation.

## 1. Environment

**Colab:** run `%pip`, restart runtime, continue. **Local:** use your venv with `datasets`, `transformers`, `tqdm`.

In [ ]:
# # Colab: skip when working locally with an existing venv.
# %pip install -q datasets transformers tqdm

## 2. Setup & paths

Resolves `REPO_ROOT`, optional Drive mount, output dirs, and the same prompt helpers as `dev.ipynb` / `submission.ipynb`.

In [1]:
import hashlib
import json
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

from tqdm.auto import tqdm


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT))

from utils import last_boxed_only_string, remove_boxed

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
PUBLIC_PATH = REPO_ROOT / "data" / "public.jsonl"
PRIVATE_PATH = REPO_ROOT / "data" / "private.jsonl"
SFT_SOURCES_DIR = REPO_ROOT / "data" / "sft_sources"
SFT_SOURCES_DIR.mkdir(parents=True, exist_ok=True)

MAX_TEMPLATE_TOKENS = 7900
MAX_TRACE_CHARS = 12_000
THINKING_OPEN = "<think>"
THINKING_CLOSE = "</think>"
THINKING_TEMPLATE = "explicit_redacted_thinking"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DRIVE_PROJECT_ROOT: Optional[Path] = None
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
except ImportError:
    print("Skip: not Google Colab — using local paths only.")

print(f"REPO_ROOT={REPO_ROOT}")
print(f"SFT_SOURCES_DIR={SFT_SOURCES_DIR}")

Skip: not Google Colab — using local paths only.
REPO_ROOT=/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition
SFT_SOURCES_DIR=/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/data/sft_sources


/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Shared helpers

Prompt construction, answer normalization, decontamination, token counting, validation, and JSONL I/O. Reused by every source section.

In [3]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FREEFORM_FINAL_RE = re.compile(r"^\\boxed\{.+\}$")
MCQ_FINAL_RE = re.compile(r"^\\boxed\{[A-J]\}$")
CJK_RE = re.compile(
    r"[\u4e00-\u9fff\u3400-\u4dbf\u3040-\u30ff\uac00-\ud7af]"
)
INLINE_MCQ_LABEL_RE = re.compile(r"\([A-J]\)", re.I)
SEQUENCE_RE = re.compile(
    r"\bsequence\b|\brecurrence\b|\ba_n\b|\ba_\{n\}|\ba_\{\s*n\s*\}", re.I
)
WEAK_TOPICS = {"number_theory", "sequence_recurrence", "geometry"}


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


def normalize_question_for_overlap(text: str) -> str:
    s = text.lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("$", "")
    s = re.sub(r"\\(?:mathrm|mathbf|text|textbf)\{([^}]*)\}", r"\1", s)
    s = re.sub(r"[^a-z0-9\\=+\-*/^_{}().,\[\]]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_public_question_keys(public_path: Path) -> set[str]:
    keys: set[str] = set()
    if not public_path.is_file():
        print(f"Warning: {public_path} missing — skipping decontamination.")
        return keys
    with open(public_path) as f:
        for line in f:
            row = json.loads(line)
            keys.add(normalize_question_for_overlap(row["question"]))
    print(f"Loaded {len(keys)} public question keys for decontamination.")
    return keys


def last_non_empty_line(text: str) -> str:
    for line in reversed(text.rstrip().splitlines()):
        if line.strip():
            return line.strip()
    return ""


def extract_boxed_answer(response: str) -> Optional[str]:
    boxed = last_boxed_only_string(response)
    if boxed is None:
        return None
    return remove_boxed(boxed)


def normalize_freeform_response(solution: str) -> tuple[Optional[str], Optional[str]]:
    """Return (response, answer) with a single final \\boxed{...} line, or (None, None)."""
    text = solution.strip()
    if not text:
        return None, None

    answer = extract_boxed_answer(text)
    if answer is None:
        return None, None

    # Drop trailing content after the last boxed answer (rare in MATH).
    boxed = last_boxed_only_string(text)
    assert boxed is not None
    idx = text.rfind(boxed)
    prefix = text[:idx].rstrip()
    final_line = boxed.strip()
    response = f"{prefix}\n{final_line}" if prefix else final_line
    return response, answer.strip()


def validate_final_line(response: str, task_type: str) -> bool:
    line = last_non_empty_line(response)
    if task_type == "mcq":
        return bool(MCQ_FINAL_RE.match(line))
    return bool(FREEFORM_FINAL_RE.match(line))


def contains_cjk(text: str) -> bool:
    return bool(CJK_RE.search(text))


def has_inline_mcq(text: str) -> bool:
    """Detect (A)...(B)...(C) option blocks embedded in problem text."""
    return len(INLINE_MCQ_LABEL_RE.findall(text)) >= 3


def split_reasoning_and_final(response: str) -> tuple[str, str]:
    final_line = last_non_empty_line(response)
    body = response.rstrip()
    if body.endswith(final_line):
        reasoning = body[: -len(final_line)].rstrip()
    else:
        reasoning = body
    return reasoning, final_line


def wrap_thinking_response(response: str) -> str:
    """Reasoning inside <think>; \\boxed{} after the closing tag (D005)."""
    reasoning, final_line = split_reasoning_and_final(response)
    parts = [THINKING_OPEN]
    if reasoning:
        parts.append(reasoning)
    parts.append(THINKING_CLOSE)
    parts.append("")
    parts.append(final_line)
    return "\n".join(parts)


def has_thinking_wrapper(response: str) -> bool:
    return THINKING_OPEN in response and THINKING_CLOSE in response


def render_training_messages(
    question: str,
    options: Optional[list],
    response: str,
) -> list[dict[str, str]]:
    system, user = build_prompt(question, options)
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": response},
    ]


def count_template_tokens(tokenizer, messages: list[dict[str, str]]) -> int:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return len(tokenizer.encode(text, add_special_tokens=False))


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


@dataclass
class PrepStats:
    source: str
    input_rows: int = 0
    ready_rows: int = 0
    reject_counts: Counter = field(default_factory=Counter)
    topic_counts: Counter = field(default_factory=Counter)
    level_counts: Counter = field(default_factory=Counter)
    trace_char_hist: Counter = field(default_factory=Counter)
    template_token_hist: Counter = field(default_factory=Counter)

    def to_dict(self) -> dict[str, Any]:
        return {
            "source": self.source,
            "input_rows": self.input_rows,
            "ready_rows": self.ready_rows,
            "reject_counts": dict(self.reject_counts),
            "topic_counts": dict(self.topic_counts),
            "level_counts": dict(self.level_counts),
            "trace_char_buckets": dict(self.trace_char_hist),
            "template_token_buckets": dict(self.template_token_hist),
        }


def bucket_value(value: int, edges: list[int]) -> str:
    for edge in edges:
        if value <= edge:
            return f"<={edge}"
    return f">{edges[-1]}"


TRACE_BUCKETS = [500, 1000, 2000, 4000, 8000]
TOKEN_BUCKETS = [512, 1024, 2048, 4096, 7900]


def validate_ready_corpus(
    rows: list[dict[str, Any]],
    source: str,
    *,
    require_thinking_wrapper: bool = False,
) -> None:
    assert rows, f"{source}: no ready rows"
    for i, row in enumerate(rows):
        assert row["source"] == source
        assert row["task_type"] in {"mcq", "freeform"}
        assert validate_final_line(row["response"], row["task_type"])
        if require_thinking_wrapper:
            assert has_thinking_wrapper(row["response"]), f"row {i}: missing thinking wrapper"
        assert row["template_tokens"] <= MAX_TEMPLATE_TOKENS
        assert str(row["answer"]).strip()
        assert not contains_cjk(row["question"])
        assert not contains_cjk(row["response"])
    print(f"Validation passed for {len(rows)} ready rows ({source}).")

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
public_question_keys = load_public_question_keys(PUBLIC_PATH)
private_question_keys = load_public_question_keys(PRIVATE_PATH)
competition_question_keys = public_question_keys | private_question_keys
print(f"Competition decont keys: {len(competition_question_keys)} (public + private)")

Loaded 1126 public question keys for decontamination.
Loaded 943 public question keys for decontamination.
Competition decont keys: 2068 (public + private)


## 4. MATH train (`EleutherAI/hendrycks_math`)

- **Split:** `train` only (7,500 rows across 7 subject configs).
- **Task type:** free-form.
- **Topic tags:** map HF config → topic; keyword-detect `sequence_recurrence`.
- **Weak-topic upweight:** handled at mix time — rows tagged `number_theory`, `geometry`, or `sequence_recurrence` get 3× weight in the final mixer.
- **Response:** MATH `solution` field (short CoT ending in `\boxed{}`).

Filters: must have `\boxed{}`, no overlap with `public.jsonl` questions, final-line regex, ≤ 7900 tokens after chat templating.

In [ ]:
from datasets import concatenate_datasets, load_dataset

MATH_SOURCE = "math_train"
MATH_DATASET_ID = "EleutherAI/hendrycks_math"
MATH_CONFIGS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

MATH_CONFIG_TO_TOPIC = {
    "algebra": "algebra",
    "counting_and_probability": "counting_probability",
    "geometry": "geometry",
    "intermediate_algebra": "intermediate_algebra",
    "number_theory": "number_theory",
    "prealgebra": "prealgebra",
    "precalculus": "precalculus",
}

MATH_READY_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_ready.jsonl"
MATH_STATS_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_stats.json"
MATH_REJECTS_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_rejects.jsonl"


def infer_math_topic(config: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    return MATH_CONFIG_TO_TOPIC[config]


def load_math_train() -> list[dict[str, Any]]:
    parts = []
    for config in MATH_CONFIGS:
        ds = load_dataset(MATH_DATASET_ID, config, split="train")
        ds = ds.add_column("hf_config", [config] * len(ds))
        parts.append(ds)
        print(f"  {config}: {len(ds)} train rows")
    merged = concatenate_datasets(parts)
    print(f"MATH train total: {len(merged)}")
    return merged


def prepare_math_row(
    row: dict[str, Any],
    *,
    row_idx: int,
) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    config = row["hf_config"]
    problem = row["problem"].strip()
    solution = row.get("solution", "")
    source_id = f"{config}:{row_idx}"

    reject_base = {
        "source": MATH_SOURCE,
        "source_id": source_id,
        "hf_config": config,
        "level": row.get("level"),
    }

    if not problem:
        return None, {**reject_base, "reason": "empty_problem"}

    qkey = normalize_question_for_overlap(problem)
    if qkey in public_question_keys:
        return None, {**reject_base, "reason": "public_overlap", "question_preview": problem[:200]}

    response, answer = normalize_freeform_response(solution)
    if response is None or answer is None:
        return None, {**reject_base, "reason": "missing_boxed", "solution_preview": str(solution)[:200]}

    if not validate_final_line(response, "freeform"):
        return None, {**reject_base, "reason": "bad_final_line", "response_tail": response[-120:]}

    topic = infer_math_topic(config, problem)
    messages = render_training_messages(problem, None, response)
    template_tokens = count_template_tokens(tokenizer, messages)
    if template_tokens > MAX_TEMPLATE_TOKENS:
        return None, {
            **reject_base,
            "reason": "too_long_tokens",
            "template_tokens": template_tokens,
        }

    ready = {
        "source": MATH_SOURCE,
        "source_id": source_id,
        "task_type": "freeform",
        "topic": topic,
        "level": row.get("level"),
        "hf_config": config,
        "weak_topic": topic in WEAK_TOPICS,
        "question": problem,
        "options": None,
        "answer": answer,
        "response": response,
        "trace_chars": len(response),
        "template_tokens": template_tokens,
    }
    return ready, None


def run_math_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print("Loading MATH train …")
    raw_rows = load_math_train()

    stats = PrepStats(source=MATH_SOURCE, input_rows=len(raw_rows))
    ready_rows: list[dict[str, Any]] = []
    rejects: list[dict[str, Any]] = []

    config_counters: dict[str, int] = defaultdict(int)
    for row in tqdm(raw_rows, desc="MATH prep"):
        config = row["hf_config"]
        idx = config_counters[config]
        config_counters[config] += 1

        ready, reject = prepare_math_row(row, row_idx=idx)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue

        ready_rows.append(ready)
        stats.topic_counts[ready["topic"]] += 1
        stats.level_counts[ready.get("level") or "unknown"] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(ready["template_tokens"], TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    return ready_rows, rejects, stats

In [ ]:
math_ready, math_rejects, math_stats = run_math_prep()

validate_ready_corpus(math_ready, MATH_SOURCE)

write_jsonl(MATH_READY_PATH, math_ready)
write_jsonl(MATH_REJECTS_PATH, math_rejects)

stats_payload = {
    **math_stats.to_dict(),
    "dataset_id": MATH_DATASET_ID,
    "configs": MATH_CONFIGS,
    "ready_path": str(MATH_READY_PATH),
    "rejects_path": str(MATH_REJECTS_PATH),
    "ready_sha256": file_sha256(MATH_READY_PATH),
    "weak_topic_rows": sum(1 for r in math_ready if r["weak_topic"]),
    "mean_trace_chars": round(sum(r["trace_chars"] for r in math_ready) / max(len(math_ready), 1), 1),
    "mean_template_tokens": round(
        sum(r["template_tokens"] for r in math_ready) / max(len(math_ready), 1), 1
    ),
    "seed": RANDOM_SEED,
}

with open(MATH_STATS_PATH, "w") as f:
    json.dump(stats_payload, f, indent=2)

print(json.dumps(stats_payload, indent=2))
print(f"\nWrote {MATH_READY_PATH}")
print(f"Wrote {MATH_REJECTS_PATH} ({len(math_rejects)} rejects)")
print(f"Wrote {MATH_STATS_PATH}")

### 4.1 Spot-check 20 ready rows

Eyeball format, reasoning quality, and topic tags before moving on.

In [ ]:
SPOT_CHECK_N = 20
spot = random.sample(math_ready, k=min(SPOT_CHECK_N, len(math_ready)))

for i, row in enumerate(spot, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | topic={row['topic']} | weak={row['weak_topic']} | "
        f"tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"][:120])
    print("Final line:", last_non_empty_line(row["response"]))
    print("Tail:", row["response"][-180:].replace("\n", " "))

### 4.2 Optional: copy artifacts to Drive

Keeps `sft_sources/` recoverable across Colab disconnects.

In [ ]:
# Run section 8 (Drive sync) after all source prep cells have run.
print("Skip: use section 8 after all source prep cells have run.")

## 5. NuminaMath-CoT — Step 2 clean rebuild (`AI-MO/NuminaMath-CoT`)

Pipeline Step 2 per `docs/sft/pipeline.md`. Writes `numina_cot_clean_{ready,stats,rejects}.jsonl`.

- **Split:** `train` (~860k rows), streamed.
- **Task type:** free-form only (inline MCQ rows **rejected**).
- **Filters:** English-only (CJK reject); `trace_chars > 2000` on raw `solution`; drop `gsm8k` / `orca_math`; decontam vs `public.jsonl` + `private.jsonl`; assistant uses explicit `<think>` wrapper (D005); `template_tokens ≤ 7900`; wrapped `trace_chars ≤ 12000`.
- **Cap:** reservoir sample up to `NUMINA_MAX_READY` qualifying rows (default 25k). `NUMINA_MAX_SCAN` limits scan for smoke tests.

Topic tags map the HF `source` field, with keyword overrides for weak topics.

In [4]:
from datasets import concatenate_datasets, load_dataset

NUMINA_SOURCE = "numina_cot_clean"
NUMINA_DATASET_ID = "AI-MO/NuminaMath-CoT"
NUMINA_DROP_SOURCES = {"gsm8k", "orca_math"}
NUMINA_MIN_TRACE_CHARS = 2000
# Reservoir cap on qualifying rows (pre-tokenization). None = keep all (~100k+).
NUMINA_MAX_READY: Optional[int] = 25_000
# Set for local smoke tests only (e.g. 50_000); None = scan full stream.
NUMINA_MAX_SCAN: Optional[int] = None

NUMINA_SOURCE_TO_TOPIC = {
    "olympiads": "olympiads",
    "aops_forum": "aops_forum",
    "amc_aime": "amc_aime",
    "cn_k12": "cn_k12",
    "math": "math",
    "synthetic_math": "synthetic_math",
    "synthetic_amc": "synthetic_amc",
    "gsm8k": "gsm8k",
    "orca_math": "orca_math",
}

NUMBER_THEORY_RE = re.compile(
    r"\b(gcd|lcm|modulo|\bmod\b|remainder|prime|divisor|congruent|coprime)\b",
    re.I,
)
GEOMETRY_RE = re.compile(
    r"\b(triangle|circle|angle|perpendicular|parallel|inradius|circumradius|polygon|circumcircle)\b",
    re.I,
)
BOXED_HINT_RE = re.compile(r"\\boxed\s*\{")

NUMINA_READY_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_ready.jsonl"
NUMINA_STATS_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_stats.json"
NUMINA_REJECTS_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_rejects.jsonl"


def infer_numina_topic(source: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    if NUMBER_THEORY_RE.search(problem):
        return "number_theory"
    if GEOMETRY_RE.search(problem):
        return "geometry"
    return NUMINA_SOURCE_TO_TOPIC.get(source, source)


def numina_qualifies(row: dict[str, Any], *, source_id: str) -> tuple[bool, Optional[dict[str, Any]]]:
    """Cheap pre-token filters. Returns (ok, reject_dict)."""
    src = row.get("source", "unknown")
    problem = (row.get("problem") or "").strip()
    solution = row.get("solution") or ""

    reject_base = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "hf_source": src,
    }

    if src in NUMINA_DROP_SOURCES:
        return False, {**reject_base, "reason": "dropped_source"}
    if not problem:
        return False, {**reject_base, "reason": "empty_problem"}
    if contains_cjk(problem) or contains_cjk(solution):
        return False, {**reject_base, "reason": "cjk_text", "question_preview": problem[:200]}
    if has_inline_mcq(problem):
        return False, {**reject_base, "reason": "inline_mcq", "question_preview": problem[:200]}
    qkey = normalize_question_for_overlap(problem)
    if qkey in competition_question_keys:
        return False, {
            **reject_base,
            "reason": "competition_overlap",
            "question_preview": problem[:200],
        }
    if len(solution.strip()) <= NUMINA_MIN_TRACE_CHARS:
        return False, {**reject_base, "reason": "short_trace", "trace_chars": len(solution.strip())}
    if not BOXED_HINT_RE.search(solution):
        return False, {**reject_base, "reason": "missing_boxed_hint", "solution_preview": str(solution)[:200]}

    return True, None


def prepare_numina_row(
    row: dict[str, Any],
    *,
    source_id: str,
) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    src = row.get("source", "unknown")
    problem = row["problem"].strip()
    solution = row.get("solution") or ""

    reject_base = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "hf_source": src,
    }

    response, answer = normalize_freeform_response(solution)
    if response is None or answer is None or not str(answer).strip():
        return None, {**reject_base, "reason": "missing_boxed", "solution_preview": str(solution)[:200]}

    if not validate_final_line(response, "freeform"):
        return None, {**reject_base, "reason": "bad_final_line", "response_tail": response[-120:]}

    response = wrap_thinking_response(response)
    if len(response) > MAX_TRACE_CHARS:
        return None, {
            **reject_base,
            "reason": "long_trace",
            "trace_chars": len(response),
        }

    topic = infer_numina_topic(src, problem)
    messages = render_training_messages(problem, None, response)
    template_tokens = count_template_tokens(tokenizer, messages)
    if template_tokens > MAX_TEMPLATE_TOKENS:
        return None, {
            **reject_base,
            "reason": "too_long_tokens",
            "template_tokens": template_tokens,
        }

    ready = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "task_type": "freeform",
        "topic": topic,
        "hf_source": src,
        "weak_topic": topic in WEAK_TOPICS,
        "question": problem,
        "options": None,
        "answer": answer,
        "response": response,
        "trace_chars": len(response),
        "template_tokens": template_tokens,
        "thinking_template": THINKING_TEMPLATE,
    }
    return ready, None


def run_numina_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print(f"Loading {NUMINA_DATASET_ID} (streaming) …")
    stream = load_dataset(NUMINA_DATASET_ID, split="train", streaming=True)

    stats = PrepStats(source=NUMINA_SOURCE)
    rejects: list[dict[str, Any]] = []
    reservoir: list[dict[str, Any]] = []
    qualifying_seen = 0

    bulk_reject_reasons = {
        "dropped_source",
        "short_trace",
        "missing_boxed_hint",
        "cjk_text",
        "inline_mcq",
    }

    for row_idx, row in enumerate(tqdm(stream, desc="Numina scan")):
        stats.input_rows += 1
        source_id = f"{row.get('source', 'unknown')}:{row_idx}"

        ok, reject = numina_qualifies(row, source_id=source_id)
        if not ok:
            reason = reject["reason"]
            stats.reject_counts[reason] += 1
            if reason not in bulk_reject_reasons:
                rejects.append(reject)
            continue

        if NUMINA_MAX_SCAN is not None and stats.input_rows >= NUMINA_MAX_SCAN:
            print(f"Stopping scan at NUMINA_MAX_SCAN={NUMINA_MAX_SCAN}")
            break

        qualifying_seen += 1
        item = dict(row)
        item["_source_id"] = source_id
        cap = NUMINA_MAX_READY
        if cap is None:
            reservoir.append(item)
            continue

        if len(reservoir) < cap:
            reservoir.append(item)
        else:
            j = random.randint(0, qualifying_seen - 1)
            if j < cap:
                reservoir[j] = item

    print(
        f"Qualifying rows: {qualifying_seen}"
        + (
            f" (reservoir cap {NUMINA_MAX_READY} → {len(reservoir)} to tokenize)"
            if NUMINA_MAX_READY
            else ""
        )
    )

    ready_rows: list[dict[str, Any]] = []
    for item in tqdm(reservoir, desc="Numina tokenize"):
        source_id = item["_source_id"]
        row = {k: v for k, v in item.items() if k != "_source_id"}

        ready, reject = prepare_numina_row(row, source_id=source_id)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue

        ready_rows.append(ready)
        stats.topic_counts[ready["topic"]] += 1
        stats.level_counts[ready.get("hf_source") or "unknown"] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(ready["template_tokens"], TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    stats.reject_counts["qualifying_seen"] = qualifying_seen
    return ready_rows, rejects, stats

In [5]:
numina_ready, numina_rejects, numina_stats = run_numina_prep()

validate_ready_corpus(numina_ready, NUMINA_SOURCE, require_thinking_wrapper=True)

write_jsonl(NUMINA_READY_PATH, numina_ready)
write_jsonl(NUMINA_REJECTS_PATH, numina_rejects)

trace_chars = [r["trace_chars"] for r in numina_ready]
template_tokens = [r["template_tokens"] for r in numina_ready]

stats_payload = {
    **numina_stats.to_dict(),
    "dataset_id": NUMINA_DATASET_ID,
    "thinking_template": THINKING_TEMPLATE,
    "ready_path": str(NUMINA_READY_PATH),
    "rejects_path": str(NUMINA_REJECTS_PATH),
    "ready_sha256": file_sha256(NUMINA_READY_PATH) if numina_ready else None,
    "weak_topic_rows": sum(1 for r in numina_ready if r["weak_topic"]),
    "mean_trace_chars": round(sum(trace_chars) / max(len(trace_chars), 1), 1),
    "median_trace_chars": sorted(trace_chars)[len(trace_chars) // 2] if trace_chars else 0,
    "p95_trace_chars": sorted(trace_chars)[int(0.95 * len(trace_chars))] if trace_chars else 0,
    "mean_template_tokens": round(sum(template_tokens) / max(len(template_tokens), 1), 1),
    "median_template_tokens": sorted(template_tokens)[len(template_tokens) // 2]
    if template_tokens
    else 0,
    "p95_template_tokens": sorted(template_tokens)[int(0.95 * len(template_tokens))]
    if template_tokens
    else 0,
    "seed": RANDOM_SEED,
    "numina_max_ready": NUMINA_MAX_READY,
    "numina_max_scan": NUMINA_MAX_SCAN,
}

with open(NUMINA_STATS_PATH, "w") as f:
    json.dump(stats_payload, f, indent=2)

print(json.dumps(stats_payload, indent=2))
print(f"\nWrote {NUMINA_READY_PATH}")
print(f"Wrote {NUMINA_REJECTS_PATH} ({len(numina_rejects)} sampled rejects)")
print(f"Wrote {NUMINA_STATS_PATH}")

Loading AI-MO/NuminaMath-CoT (streaming) …


Numina scan: 859494it [02:17, 6242.79it/s]


Qualifying rows: 99826 (reservoir cap 25000 → 25000 to tokenize)


Numina tokenize: 100%|██████████| 25000/25000 [00:41<00:00, 603.67it/s]


Validation passed for 23089 ready rows (numina_cot_clean).
{
  "source": "numina_cot_clean",
  "input_rows": 859494,
  "ready_rows": 23089,
  "reject_counts": {
    "short_trace": 535762,
    "dropped_source": 160656,
    "inline_mcq": 47723,
    "missing_boxed_hint": 13396,
    "cjk_text": 2124,
    "competition_overlap": 7,
    "missing_boxed": 1620,
    "bad_final_line": 291,
    "qualifying_seen": 99826
  },
  "topic_counts": {
    "aops_forum": 1524,
    "olympiads": 10828,
    "cn_k12": 1981,
    "sequence_recurrence": 1761,
    "geometry": 5632,
    "number_theory": 1203,
    "amc_aime": 36,
    "math": 59,
    "synthetic_amc": 44,
    "synthetic_math": 21
  },
  "level_counts": {
    "aops_forum": 2571,
    "olympiads": 17181,
    "cn_k12": 3084,
    "amc_aime": 66,
    "math": 102,
    "synthetic_amc": 54,
    "synthetic_math": 31
  },
  "trace_char_buckets": {
    "<=4000": 22430,
    "<=8000": 282,
    "<=2000": 327,
    "<=500": 10,
    "<=1000": 37,
    ">8000": 3
  },
  "

### 5.1 Run clean prep + spot-check

Run the cell below first (streams Numina — long on first full run). Then spot-check 20 rows.

In [6]:
SPOT_CHECK_N = 20
spot_numina = random.sample(numina_ready, k=min(SPOT_CHECK_N, len(numina_ready)))

for i, row in enumerate(spot_numina, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | hf={row.get('hf_source')} | topic={row['topic']} | "
        f"weak={row['weak_topic']} | tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"][:120])
    print("Final line:", last_non_empty_line(row["response"]))
    print("Thinking wrapper:", has_thinking_wrapper(row["response"]))
    print("Tail:", row["response"][-220:].replace("\n", " "))

[1] olympiads:89533 | hf=olympiads | topic=olympiads | weak=False | tokens=966 | chars=2055
Q: Determine the positive integer solutions of the following equation:  $$ xy z + 2x + 3y + 6z = xy + 2xz + 3yz $$ …
A: (x, y, z) = (4, 3, 1)
Final line: \boxed{(x, y, z) = (4, 3, 1)}
Thinking wrapper: True
Tail:     \]  7. **Consider all valid positive integer solutions:**        Upon validation and correction of all combinations, we get:     \( (x, y, z) = (4, 3, 1) \)  ### Conclusion  \[ </think>  \boxed{(x, y, z) = (4, 3, 1)}
[2] cn_k12:342873 | hf=cn_k12 | topic=cn_k12 | weak=False | tokens=1169 | chars=2670
Q: Given that $\sin(3\pi - \alpha) = 2\sin\left(\frac{\pi}{2} + \alpha\right)$, find the value of $\frac{\sin^3(\pi - \alpha) - \sin\left(\frac{\pi}{2} - \alpha\right)}{3\cos\left(\frac{\pi}{2} + \alpha\right) + 2\cos(\pi + \alpha)}$. …
A: \frac{8 - 5\sqrt{5}}{40\sqrt{5}}
Final line: \boxed{\frac{8 - 5\sqrt{5}}{40\sqrt{5}}}
Thinking wrapper: True
Tail: } - \frac{\sqrt{5}}{\sqrt{5}}}{8} 

### 5.2 Corpus audit

Re-audit **all** ready rows (in-memory `numina_ready` or `numina_cot_clean_ready.jsonl` on disk). Run after §5.1. Use output to decide whether to rebuild filters, raise `NUMINA_MAX_READY`, or downsample at Step 5.

In [7]:
import statistics

BOXED_IN_THINKING_RE = re.compile(r"\\boxed\s*\{")
DANGLING_CLOSE_RE = re.compile(r"(Conclusion:\s*\\\[|\\\[\s*)$")


def _numina_thinking_body(response: str) -> str:
    if THINKING_CLOSE not in response:
        return ""
    return response.split(THINKING_CLOSE, 1)[0].split(THINKING_OPEN, 1)[-1]


def _numina_final_line(response: str) -> str:
    after = response.split(THINKING_CLOSE, 1)[-1].strip()
    return after.splitlines()[-1] if after else ""


def audit_numina_corpus(rows: list[dict[str, Any]]) -> dict[str, Any]:
    n = len(rows)
    if n == 0:
        return {"n": 0}

    trace_chars = [r["trace_chars"] for r in rows]
    template_tokens = [r["template_tokens"] for r in rows]
    hf_sources: Counter = Counter()
    topics: Counter = Counter()

    issues: Counter = Counter()
    letter_final_rows: list[dict[str, Any]] = []
    short_rows: list[dict[str, Any]] = []
    multi_boxed_rows: list[dict[str, Any]] = []

    for r in rows:
        q, resp = r["question"], r["response"]
        hf_sources[r.get("hf_source", "?")] += 1
        topics[r["topic"]] += 1

        if not has_thinking_wrapper(resp):
            issues["missing_thinking_wrapper"] += 1
        fl = _numina_final_line(resp)
        if not validate_final_line(resp, "freeform"):
            issues["bad_final_regex"] += 1
        if contains_cjk(q) or contains_cjk(resp):
            issues["cjk_leak"] += 1
        if has_inline_mcq(q):
            issues["inline_mcq_leak"] += 1
        if normalize_question_for_overlap(q) in competition_question_keys:
            issues["competition_overlap"] += 1
        if r["template_tokens"] > MAX_TEMPLATE_TOKENS:
            issues["over_token_cap"] += 1
        if r["trace_chars"] > MAX_TRACE_CHARS:
            issues["over_char_cap"] += 1

        tb = _numina_thinking_body(resp)
        if BOXED_IN_THINKING_RE.search(tb):
            issues["extra_boxed_in_thinking"] += 1
            if len(multi_boxed_rows) < 5:
                multi_boxed_rows.append(r)
        if DANGLING_CLOSE_RE.search(tb.rstrip()[-80:]):
            issues["dangling_latex_before_close"] += 1
        if len(tb.strip()) < 100:
            issues["thin_thinking"] += 1
        if r["trace_chars"] < NUMINA_MIN_TRACE_CHARS:
            issues["post_wrap_under_min_trace"] += 1
            short_rows.append(r)
        if MCQ_FINAL_RE.match(fl):
            issues["letter_final_mcq_style"] += 1
            if len(letter_final_rows) < 5:
                letter_final_rows.append(r)

    def pct(x: int) -> float:
        return 100.0 * x / n

    return {
        "n": n,
        "trace_chars": trace_chars,
        "template_tokens": template_tokens,
        "hf_sources": hf_sources,
        "topics": topics,
        "issues": issues,
        "letter_final_rows": letter_final_rows,
        "short_rows": short_rows,
        "multi_boxed_rows": multi_boxed_rows,
        "weak_topic_rows": sum(1 for r in rows if r.get("weak_topic")),
        "pct": pct,
    }


# Load corpus: prefer in-memory from §5.1, else disk
try:
    _audit_rows = numina_ready
    _audit_source = "in-memory numina_ready"
except NameError:
    if NUMINA_READY_PATH.is_file():
        _audit_rows = read_jsonl(NUMINA_READY_PATH)
        _audit_source = str(NUMINA_READY_PATH)
    else:
        _audit_rows = []
        _audit_source = "(none — run §5.1 first)"
if not _audit_rows and NUMINA_READY_PATH.is_file():
    _audit_rows = read_jsonl(NUMINA_READY_PATH)
    _audit_source = str(NUMINA_READY_PATH)

report = audit_numina_corpus(_audit_rows)
n = report["n"]
pct = report["pct"]

print(f"Audit source: {_audit_source}")
print(f"Rows: {n:,}")
if n == 0:
    raise SystemExit("No rows to audit.")

tc = report["trace_chars"]
tt = report["template_tokens"]
print(
    f"trace_chars: mean={statistics.mean(tc):.0f} median={statistics.median(tc):.0f} "
    f"min={min(tc)} max={max(tc)} p5={sorted(tc)[max(0, int(0.05 * n) - 1)]} "
    f"p95={sorted(tc)[min(n - 1, int(0.95 * n))]}"
)
print(
    f"template_tokens: mean={statistics.mean(tt):.0f} median={statistics.median(tt):.0f} "
    f"max={max(tt)}"
)
print(f"weak_topic: {report['weak_topic_rows']:,} ({pct(report['weak_topic_rows']):.1f}%)")
print()

if NUMINA_STATS_PATH.is_file():
    saved = json.loads(NUMINA_STATS_PATH.read_text())
    inp = saved.get("input_rows", 0)
    qual = saved.get("reject_counts", {}).get("qualifying_seen", 0)
    cap = saved.get("numina_max_ready")
    print("Funnel (from numina_cot_clean_stats.json)")
    print(f"  scanned: {inp:,}")
    print(f"  qualifying (pre-token): {qual:,}" + (f" ({100 * qual / inp:.1f}% of scan)" if inp else ""))
    print(f"  ready: {n:,}" + (f" ({100 * n / qual:.1f}% of qualifying)" if qual else ""))
    if cap is not None and qual > cap:
        est = int(qual * n / min(cap, qual))
        print(
            f"  NUMINA_MAX_READY={cap:,} capped tokenization — only {min(cap, qual):,} rows tokenized."
        )
        print(f"  Estimated ready if all qualifying tokenized (~{n / min(cap, qual):.1%} pass): ~{est:,}")
    print("  Scan rejects:")
    for reason, cnt in sorted(
        ((k, v) for k, v in saved.get("reject_counts", {}).items() if k != "qualifying_seen"),
        key=lambda x: -x[1],
    )[:8]:
        print(f"    {reason}: {cnt:,}")
    print()

print("HF source (top 5):")
for k, v in report["hf_sources"].most_common(5):
    print(f"  {k}: {v:,} ({pct(v):.1f}%)")
print()
print("Quality issues (re-audit; 0 = pass):")
for key in sorted(report["issues"]):
    c = report["issues"][key]
    print(f"  {key}: {c:,} ({pct(c):.1f}%)")
print()

# Step 4 gate: >2 hard failures → rebuild
hard = (
    report["issues"]["missing_thinking_wrapper"]
    + report["issues"]["bad_final_regex"]
    + report["issues"]["cjk_leak"]
    + report["issues"]["inline_mcq_leak"]
    + report["issues"]["competition_overlap"]
    + sum(1 for r in report["short_rows"] if r["trace_chars"] < 500)
)
print(f"Step 4 hard-failure count (heuristic): {hard}  (pipeline: rebuild if > 2)")
if report["issues"]["letter_final_mcq_style"]:
    print(
        f"  letter_final_mcq_style: {report['issues']['letter_final_mcq_style']:,} — "
        "consider rejecting \\boxed{{A-J}} free-form finals"
    )
print()

if report["letter_final_rows"]:
    print("Sample letter-final rows:")
    for r in report["letter_final_rows"][:3]:
        print(f"  {r['source_id']} | final={_numina_final_line(r['response'])}")
        print(f"    Q: {r['question'][:140].replace(chr(10), ' ')}…")
    print()

if report["short_rows"]:
    print("Shortest rows (post-wrap trace_chars < NUMINA_MIN_TRACE_CHARS):")
    for r in sorted(report["short_rows"], key=lambda x: x["trace_chars"])[:5]:
        print(
            f"  {r['trace_chars']} chars | {r['template_tokens']} tok | {r['source_id']} | "
            f"final={_numina_final_line(r['response'])[:50]}"
        )
    print()

if report["multi_boxed_rows"]:
    print("Sample extra-\\boxed in thinking:")
    for r in report["multi_boxed_rows"][:2]:
        tb = _numina_thinking_body(r["response"])
        print(
            f"  {r['source_id']} | boxes_in_think={len(BOXED_IN_THINKING_RE.findall(tb))} | "
            f"final={_numina_final_line(r['response'])[:60]}"
        )

Audit source: in-memory numina_ready
Rows: 23,089
trace_chars: mean=2507 median=2395 min=148 max=9160 p5=2037 p95=3376
template_tokens: mean=1102 median=1066 max=3731
weak_topic: 8,596 (37.2%)

Funnel (from numina_cot_clean_stats.json)
  scanned: 859,494
  qualifying (pre-token): 99,826 (11.6% of scan)
  ready: 23,089 (23.1% of qualifying)
  NUMINA_MAX_READY=25,000 capped tokenization — only 25,000 rows tokenized.
  Estimated ready if all qualifying tokenized (~92.4% pass): ~92,195
  Scan rejects:
    short_trace: 535,762
    dropped_source: 160,656
    inline_mcq: 47,723
    missing_boxed_hint: 13,396
    cjk_text: 2,124
    missing_boxed: 1,620
    bad_final_line: 291
    competition_overlap: 7

HF source (top 5):
  olympiads: 17,181 (74.4%)
  cn_k12: 3,084 (13.4%)
  aops_forum: 2,571 (11.1%)
  math: 102 (0.4%)
  amc_aime: 66 (0.3%)

Quality issues (re-audit; 0 = pass):
  dangling_latex_before_close: 14,582 (63.2%)
  extra_boxed_in_thinking: 3,722 (16.1%)
  letter_final_mcq_style: 41

### 5.3 Long-biased second pass (outside 25k reservoir)

The first `numina_cot_clean_ready.jsonl` run capped at `NUMINA_MAX_READY=25_000` reservoir rows (~23k tokenized). Long traces are rare in Numina (~1–2% of qualifying rows), so uniform reservoir sampling missed most of them.

This pass **re-scans** `AI-MO/NuminaMath-CoT`, skips `source_id`s already in the first ready file, applies the usual cheap filters (`numina_qualifies`, raw `solution ≥ 2000`), keeps the **top ~25k candidates by raw solution length** in a heap, then tokenizes longest-first until **~3000** accepted rows.

Output: `data/sft_sources/numina_cot_clean_ready_long.jsonl` → `build_sft_corpus.py long-trace --ready-path …`.

Local CLI: `python scripts/build_numina_ready_long.py`

In [8]:
import heapq

NUMINA_LONG_SOURCE = "numina_cot_clean_long"
NUMINA_LONG_CANDIDATE_HEAP = 25_000  # top raw-length rows outside first pass
NUMINA_LONG_MAX_READY = 3000
NUMINA_LONG_READY_PATH = SFT_SOURCES_DIR / "numina_cot_clean_ready_long.jsonl"
NUMINA_LONG_STATS_PATH = SFT_SOURCES_DIR / "numina_cot_clean_long_stats.json"


def run_numina_long_second_pass(
    *,
    existing_source_ids: set[str],
    candidate_heap: int = NUMINA_LONG_CANDIDATE_HEAP,
    max_ready: int = NUMINA_LONG_MAX_READY,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    if not NUMINA_READY_PATH.is_file():
        raise FileNotFoundError(
            f"Missing first-pass ready file: {NUMINA_READY_PATH}. Run §5.1 first."
        )

    print(f"Loading {NUMINA_DATASET_ID} (streaming) — long second pass …")
    print(f"Skip {len(existing_source_ids):,} source_ids from first pass")
    stream = load_dataset(NUMINA_DATASET_ID, split="train", streaming=True)

    stats = PrepStats(source=NUMINA_LONG_SOURCE)
    rejects: list[dict[str, Any]] = []
    heap: list[tuple[int, int, dict[str, Any]]] = []
    tie = 0

    bulk_reject_reasons = {
        "dropped_source",
        "short_trace",
        "missing_boxed_hint",
        "cjk_text",
        "inline_mcq",
        "already_in_first_pass",
    }

    for row_idx, row in enumerate(tqdm(stream, desc="Numina long scan")):
        stats.input_rows += 1
        source_id = f"{row.get('source', 'unknown')}:{row_idx}"

        if source_id in existing_source_ids:
            stats.reject_counts["already_in_first_pass"] += 1
            continue

        ok, reject = numina_qualifies(row, source_id=source_id)
        if not ok:
            reason = reject["reason"]
            stats.reject_counts[reason] += 1
            if reason not in bulk_reject_reasons:
                rejects.append(reject)
            continue

        solution = (row.get("solution") or "").strip()
        raw_len = len(solution)
        item = dict(row)
        item["_source_id"] = source_id
        entry = (raw_len, tie, item)
        tie += 1
        if len(heap) < candidate_heap:
            heapq.heappush(heap, entry)
        elif raw_len > heap[0][0]:
            heapq.heapreplace(heap, entry)

    candidates = sorted(heap, key=lambda x: x[0], reverse=True)
    print(
        f"Qualifying outside first pass: {len(candidates):,} "
        f"(heap cap {candidate_heap:,}); tokenizing longest-first …"
    )

    ready_rows: list[dict[str, Any]] = []
    for raw_len, _, item in tqdm(candidates, desc="Numina long tokenize"):
        if len(ready_rows) >= max_ready:
            break
        source_id = item["_source_id"]
        row = {k: v for k, v in item.items() if not k.startswith("_")}
        ready, reject = prepare_numina_row(row, source_id=source_id)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue
        ready["source"] = NUMINA_LONG_SOURCE
        ready["raw_solution_chars"] = raw_len
        ready_rows.append(ready)
        stats.topic_counts[ready["topic"]] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(ready["template_tokens"], TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    trace_chars = [r["trace_chars"] for r in ready_rows]
    template_tokens = [r["template_tokens"] for r in ready_rows]
    ge6000 = sum(1 for t in trace_chars if t >= 6000)

    payload = {
        **stats.to_dict(),
        "dataset_id": NUMINA_DATASET_ID,
        "thinking_template": THINKING_TEMPLATE,
        "ready_path": str(NUMINA_LONG_READY_PATH),
        "first_pass_ready": str(NUMINA_READY_PATH),
        "first_pass_skip_ids": len(existing_source_ids),
        "candidate_heap_cap": candidate_heap,
        "candidates_tokenized": len(candidates),
        "max_ready": max_ready,
        "wrapped_trace_chars_ge_6000": ge6000,
        "mean_trace_chars": round(sum(trace_chars) / max(len(trace_chars), 1), 1),
        "median_trace_chars": sorted(trace_chars)[len(trace_chars) // 2] if trace_chars else 0,
        "p95_trace_chars": sorted(trace_chars)[int(0.95 * len(trace_chars))] if trace_chars else 0,
        "mean_template_tokens": round(sum(template_tokens) / max(len(template_tokens), 1), 1),
        "ready_sha256": None,
        "seed": RANDOM_SEED,
    }
    return ready_rows, payload


# --- run (requires §5.1 ready on disk + tokenizer from §3) ---
RUN_NUMINA_LONG_PASS = False

if RUN_NUMINA_LONG_PASS:
    _existing = {r["source_id"] for r in read_jsonl(NUMINA_READY_PATH)}
    numina_long_ready, numina_long_stats = run_numina_long_second_pass(
        existing_source_ids=_existing,
    )
    validate_ready_corpus(
        numina_long_ready, NUMINA_LONG_SOURCE, require_thinking_wrapper=True
    )
    write_jsonl(NUMINA_LONG_READY_PATH, numina_long_ready)
    numina_long_stats["ready_sha256"] = file_sha256(NUMINA_LONG_READY_PATH)
    NUMINA_LONG_STATS_PATH.write_text(json.dumps(numina_long_stats, indent=2) + "\n")
    print(json.dumps(numina_long_stats, indent=2))
    print(f"Wrote {NUMINA_LONG_READY_PATH} ({len(numina_long_ready)} rows)")
else:
    print("Set RUN_NUMINA_LONG_PASS = True to stream HF and build numina_cot_clean_ready_long.jsonl")

Set RUN_NUMINA_LONG_PASS = True to stream HF and build numina_cot_clean_ready_long.jsonl


## 6. AGIEval-Math + GaoKao-MCQ

MCQ format coverage for SFT. Combines three HuggingFace `hails/agieval-*` subsets (825 rows total):

| Subset | Dataset | Rows | Original options |
|--------|---------|------|------------------|
| GaoKao | `hails/agieval-gaokao-mathqa` | 351 | 4 |
| SAT math | `hails/agieval-sat-math` | 220 | 4 |
| AQuA-RAT | `hails/agieval-aqua-rat` | 254 | 5 |

**Note:** `hails/agieval-math` is free-form QA (no choices) and is excluded here.

- **Task type:** MCQ only.
- **10-option expansion:** sample wrong choices from sibling problems in the same subset, shuffle, remap gold letter.
- **Response:** synthetic short CoT ending in `\boxed{<letter>}` (sources ship without reasoning traces).
- **Filters:** decontamination vs `public.jsonl`, final-line regex, ≤ 7900 tokens after templating.

In [ ]:
AGIEVAL_MCQ_SOURCE = "agieval_mcq"
TARGET_MCQ_OPTIONS = 10
CHOICE_LABEL_RE = re.compile(r"^\([A-J]\)\s*")

AGIEVAL_MCQ_DATASETS = [
    {
        "dataset_id": "hails/agieval-gaokao-mathqa",
        "hf_subset": "gaokao",
        "split": "test",
        "topic": "gaokao",
    },
    {
        "dataset_id": "hails/agieval-sat-math",
        "hf_subset": "sat_math",
        "split": "test",
        "topic": "sat_math",
    },
    {
        "dataset_id": "hails/agieval-aqua-rat",
        "hf_subset": "aqua_rat",
        "split": "test",
        "topic": "aqua_rat",
    },
]

AGIEVAL_MCQ_READY_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_ready.jsonl"
AGIEVAL_MCQ_STATS_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_stats.json"
AGIEVAL_MCQ_REJECTS_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_rejects.jsonl"


def strip_choice_label(text: str) -> str:
    return CHOICE_LABEL_RE.sub("", text.strip()).strip()


def parse_hails_query_question(query: str) -> str:
    q = query.strip()
    if q.startswith("Q:"):
        q = q[2:].strip()
        if " Answer Choices:" in q:
            q = q.split(" Answer Choices:")[0].strip()
        elif "\nA:" in q:
            q = q.split("\nA:")[0].strip()
    elif q.startswith("问题："):
        q = q.split("问题：", 1)[1]
        for sep in ["\n 选项：", "\n选项：", "选项："]:
            if sep in q:
                q = q.split(sep)[0].strip()
                break
        q = q.split("\n答案：")[0].strip()
    return q.strip()


def gold_to_letter(gold: list, n_choices: int) -> Optional[str]:
    if not gold:
        return None
    idx = int(gold[0])
    if idx < 0 or idx >= n_choices:
        return None
    return chr(65 + idx)


def synthesize_mcq_response(correct_letter: str) -> str:
    return (
        "I'll solve this step by step and compare the answer choices.\n\n"
        "Working through the problem, I eliminate inconsistent options "
        "and verify the remaining candidate against the constraints.\n\n"
        f"The correct choice is {correct_letter}.\n"
        f"\\boxed{{{correct_letter}}}"
    )


def expand_mcq_options(
    options: list[str],
    correct_idx: int,
    distractor_pool: list[str],
    *,
    target: int,
    rng: random.Random,
) -> tuple[Optional[list[str]], Optional[str], Optional[str]]:
    if correct_idx < 0 or correct_idx >= len(options):
        return None, None, "bad_correct_idx"

    needed = max(0, target - len(options))
    combined = list(options)
    if needed:
        existing = set(options)
        distractors: list[str] = []
        candidates = [d for d in distractor_pool if d not in existing]
        rng.shuffle(candidates)
        for candidate in candidates:
            if len(distractors) >= needed:
                break
            if candidate not in existing:
                distractors.append(candidate)
                existing.add(candidate)
        if len(options) + len(distractors) < target:
            return None, None, "insufficient_distractors"
        combined = options + distractors

    perm = list(range(len(combined)))
    rng.shuffle(perm)
    new_options = [combined[i] for i in perm]
    new_correct_idx = perm.index(correct_idx)
    new_letter = chr(65 + new_correct_idx)
    return new_options, new_letter, None


def infer_agieval_topic(hf_subset: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    return hf_subset


def load_agieval_mcq_raw() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for spec in AGIEVAL_MCQ_DATASETS:
        ds = load_dataset(spec["dataset_id"], split=spec["split"])
        print(f"  {spec['hf_subset']}: {len(ds)} rows from {spec['dataset_id']}")
        for idx, row in enumerate(ds):
            rows.append(
                {
                    "hf_subset": spec["hf_subset"],
                    "topic_default": spec["topic"],
                    "row_idx": idx,
                    "query": row["query"],
                    "choices": row.get("choices") or [],
                    "gold": row.get("gold") or [],
                }
            )
    print(f"AGIEval/GaoKao MCQ total: {len(rows)}")
    return rows


def normalize_agieval_row(row: dict[str, Any]) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    hf_subset = row["hf_subset"]
    row_idx = row["row_idx"]
    source_id = f"{hf_subset}:{row_idx}"
    reject_base = {
        "source": AGIEVAL_MCQ_SOURCE,
        "source_id": source_id,
        "hf_subset": hf_subset,
    }

    choices_raw = row.get("choices") or []
    if not choices_raw:
        return None, {**reject_base, "reason": "missing_choices"}

    options = [strip_choice_label(c) for c in choices_raw]
    if not all(options):
        return None, {**reject_base, "reason": "empty_choice_text"}

    question = parse_hails_query_question(row["query"])
    if not question:
        return None, {**reject_base, "reason": "empty_problem"}

    letter = gold_to_letter(row.get("gold") or [], len(options))
    if letter is None:
        return None, {**reject_base, "reason": "bad_gold_index", "gold": row.get("gold")}

    correct_idx = ord(letter) - 65
    return {
        "source_id": source_id,
        "hf_subset": hf_subset,
        "question": question,
        "options": options,
        "correct_idx": correct_idx,
        "correct_letter": letter,
        "original_n_options": len(options),
    }, None


def run_agieval_mcq_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print("Loading AGIEval/GaoKao MCQ sources …")
    raw_rows = load_agieval_mcq_raw()
    stats = PrepStats(source=AGIEVAL_MCQ_SOURCE, input_rows=len(raw_rows))
    rejects: list[dict[str, Any]] = []

    normalized: list[dict[str, Any]] = []
    by_subset: dict[str, list[dict[str, Any]]] = defaultdict(list)

    for row in raw_rows:
        norm, reject = normalize_agieval_row(row)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue
        normalized.append(norm)
        by_subset[norm["hf_subset"]].append(norm)

    distractor_pools = {
        subset: [
            opt
            for item in items
            for i, opt in enumerate(item["options"])
            if i != item["correct_idx"]
        ]
        for subset, items in by_subset.items()
    }

    ready_rows: list[dict[str, Any]] = []
    for norm in tqdm(normalized, desc="AGIEval MCQ prep"):
        reject_base = {
            "source": AGIEVAL_MCQ_SOURCE,
            "source_id": norm["source_id"],
            "hf_subset": norm["hf_subset"],
        }

        qkey = normalize_question_for_overlap(norm["question"])
        if qkey in public_question_keys:
            reject = {
                **reject_base,
                "reason": "public_overlap",
                "question_preview": norm["question"][:200],
            }
            rejects.append(reject)
            stats.reject_counts["public_overlap"] += 1
            continue

        rng = random.Random(RANDOM_SEED + hash(norm["source_id"]))
        expanded_options, answer, err = expand_mcq_options(
            norm["options"],
            norm["correct_idx"],
            distractor_pools[norm["hf_subset"]],
            target=TARGET_MCQ_OPTIONS,
            rng=rng,
        )
        if err:
            reject = {
                **reject_base,
                "reason": err,
                "original_n_options": norm["original_n_options"],
            }
            rejects.append(reject)
            stats.reject_counts[err] += 1
            continue

        response = synthesize_mcq_response(answer)
        if not validate_final_line(response, "mcq"):
            reject = {
                **reject_base,
                "reason": "bad_final_line",
                "response_tail": response[-120:],
            }
            rejects.append(reject)
            stats.reject_counts["bad_final_line"] += 1
            continue

        topic = infer_agieval_topic(norm["hf_subset"], norm["question"])
        messages = render_training_messages(norm["question"], expanded_options, response)
        template_tokens = count_template_tokens(tokenizer, messages)
        if template_tokens > MAX_TEMPLATE_TOKENS:
            reject = {
                **reject_base,
                "reason": "too_long_tokens",
                "template_tokens": template_tokens,
            }
            rejects.append(reject)
            stats.reject_counts["too_long_tokens"] += 1
            continue

        ready = {
            "source": AGIEVAL_MCQ_SOURCE,
            "source_id": norm["source_id"],
            "task_type": "mcq",
            "topic": topic,
            "hf_subset": norm["hf_subset"],
            "weak_topic": topic in WEAK_TOPICS,
            "question": norm["question"],
            "options": expanded_options,
            "answer": answer,
            "response": response,
            "trace_chars": len(response),
            "template_tokens": template_tokens,
            "original_n_options": norm["original_n_options"],
            "expanded_to": TARGET_MCQ_OPTIONS,
        }
        ready_rows.append(ready)
        stats.topic_counts[topic] += 1
        stats.level_counts[norm["hf_subset"]] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(template_tokens, TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    return ready_rows, rejects, stats

In [ ]:
agieval_mcq_ready, agieval_mcq_rejects, agieval_mcq_stats = run_agieval_mcq_prep()

validate_ready_corpus(agieval_mcq_ready, AGIEVAL_MCQ_SOURCE)

write_jsonl(AGIEVAL_MCQ_READY_PATH, agieval_mcq_ready)
write_jsonl(AGIEVAL_MCQ_REJECTS_PATH, agieval_mcq_rejects)

agieval_stats_payload = {
    **agieval_mcq_stats.to_dict(),
    "datasets": AGIEVAL_MCQ_DATASETS,
    "target_mcq_options": TARGET_MCQ_OPTIONS,
    "ready_path": str(AGIEVAL_MCQ_READY_PATH),
    "rejects_path": str(AGIEVAL_MCQ_REJECTS_PATH),
    "ready_sha256": file_sha256(AGIEVAL_MCQ_READY_PATH),
    "mean_trace_chars": round(
        sum(r["trace_chars"] for r in agieval_mcq_ready) / max(len(agieval_mcq_ready), 1), 1
    ),
    "mean_template_tokens": round(
        sum(r["template_tokens"] for r in agieval_mcq_ready) / max(len(agieval_mcq_ready), 1), 1
    ),
    "seed": RANDOM_SEED,
}

with open(AGIEVAL_MCQ_STATS_PATH, "w") as f:
    json.dump(agieval_stats_payload, f, indent=2)

print(json.dumps(agieval_stats_payload, indent=2))
print(f"\nWrote {AGIEVAL_MCQ_READY_PATH}")
print(f"Wrote {AGIEVAL_MCQ_REJECTS_PATH} ({len(agieval_mcq_rejects)} rejects)")
print(f"Wrote {AGIEVAL_MCQ_STATS_PATH}")


### 6.1 Spot-check 20 ready rows

In [ ]:
spot_agieval = random.sample(agieval_mcq_ready, k=min(SPOT_CHECK_N, len(agieval_mcq_ready)))

for i, row in enumerate(spot_agieval, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | subset={row.get('hf_subset')} | topic={row['topic']} | "
        f"n_opts={len(row['options'])} (from {row['original_n_options']}) | "
        f"tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"])
    print("Options[:3]:", row["options"][:3])
    print("Final line:", last_non_empty_line(row["response"]))

## 7. Step 5 — base corpus (Numina-only, 15k)

Build `data/sft_corpus.jsonl` from `numina_cot_clean_ready.jsonl`:

- Drop `trace_chars < 500` (audit: 10 rows)
- Drop `letter_final_mcq_style` (`\\boxed{A-J}` on free-form rows)
- Weighted downsample: weak topics get **3×** weight
- Default target **15,000** rows

```bash
python scripts/build_sft_corpus.py base --target 15000 --seed 42
```

## 8. Step 5b — targeted supplements + `sft_corpus_v2` (sft-001)

Extends `scripts/build_sft_corpus.py` ([`docs/sft/pipeline.md`](../docs/sft/pipeline.md) Step 5a–5c):

| Artifact | ~rows | Notes |
|----------|------:|-------|
| `data/sft_sources/numina_cot_clean_ready_long.jsonl` | 3,000 | §5.3: top-25k raw-length Numina rows outside first-pass `source_id`s, tokenized longest-first. |
| `data/sft_sources/numina_long_trace.jsonl` | 1,500 | Sampled from long ready pool (p95 `trace_chars` ~5594; only ~53 Numina rows wrap to ≥6000). |
| `data/sft_sources/numina_multi_blank_synth.jsonl` | 1,500 | Multi-blank prompt path matches `full_public.ipynb` / pub-002; composed 2–4 part rows (native finals are single-boxed after Numina prep). |
| `data/sft_corpus_v2.jsonl` | 18,000 | `shuffle(base + supplements, seed=42)` |

```bash
# After §5.3 long ready pass:
python scripts/build_sft_corpus.py long-trace \
  --ready-path data/sft_sources/numina_cot_clean_ready_long.jsonl --target 1500
python scripts/build_sft_corpus.py merge-v2 --seed 42

# Full supplements (multi-blank unchanged):
python scripts/build_sft_corpus.py supplements --seed 42
```

Spot-check ~30 supplement rows (15 long-trace, 15 multi-blank) before training.

## 9. OpenR1-Math-220k — 1k probe (`sft-002a`, PRIMARY)

Primary SFT corpus per [`docs/sft/pipeline.md`](../docs/sft/pipeline.md) Phase 1. DeepSeek-R1 distilled traces from `open-r1/OpenR1-Math-220k` config `default`.

- **Filters:** `correctness_math_verify == True`; first qualifying generation; `len(response) ≤ 10_000` chars; `len(response) ≥ 2000`; English-only; no figure-dependent prompts; decontam vs `public.jsonl` + `private.jsonl` (substring on first 200 chars + full normalized key); `template_tokens ≤ 7900`; parseable `\boxed{}` after `</think>` (D005).
- **Sample:** shuffle seed 42 → 1000 rows.
- **Output:** `data/sft_corpus_openr1_1k.jsonl` + `data/sft_corpus_openr1_1k_manifest.json`.

Run §9.1–§9.3 below before `sft_train.ipynb` with the OpenR1 corpus path.

### 9.1 Future sources (not yet implemented)

1. **s1K-1.1** (`simplescaling/s1K-1.1`) — sft-003 fallback if OpenR1 probe is flat.
2. **Baseline-self-distilled format-fix** — external problems only, rewrite final line.

In [5]:
import subprocess
import sys

_corpus_script = REPO_ROOT / "scripts" / "build_sft_corpus.py"

# Set RUN_V2_SUPPLEMENTS = True after base corpus exists to build sft_corpus_v2.
RUN_BASE_CORPUS = True
RUN_V2_SUPPLEMENTS = False

if RUN_BASE_CORPUS:
    subprocess.run(
        [sys.executable, str(_corpus_script), "base", "--target", "15000", "--seed", "42"],
        check=True,
        cwd=REPO_ROOT,
    )
    _manifest = json.loads((REPO_ROOT / "data" / "sft_corpus_manifest.json").read_text())
    print(f"Base corpus rows: {_manifest['final_row_count']:,}")
    print(f"Base sha256: {_manifest['corpus_sha256']}")

if RUN_V2_SUPPLEMENTS:
    subprocess.run(
        [sys.executable, str(_corpus_script), "supplements", "--seed", "42"],
        check=True,
        cwd=REPO_ROOT,
    )
    _v2 = json.loads((REPO_ROOT / "data" / "sft_corpus_v2_manifest.json").read_text())
    print(f"v2 corpus rows: {_v2['final_row_count']:,}")
    print(f"v2 sha256: {_v2['corpus_sha256']}")

{
  "thinking_template": "explicit_redacted_thinking",
  "sources": [
    {
      "path": "data/sft_sources/numina_cot_clean_ready.jsonl",
      "sha256": "bbdb1c7b29f0536177bf583f6e2e78f1beea3e1d4d4472e21d1d8518949fa343",
      "source": "numina_cot_clean"
    }
  ],
  "numina_stats_ready_sha256": "bbdb1c7b29f0536177bf583f6e2e78f1beea3e1d4d4472e21d1d8518949fa343",
  "input_ready_rows": 23089,
  "after_filter_rows": 22663,
  "filter_drops": {
    "letter_final_mcq_style": 416,
    "trace_chars_below_min": 10
  },
  "min_trace_chars": 500,
  "drop_letter_final_mcq": true,
  "weak_topic_weight": 3.0,
  "sample_seed": 42,
  "target_n": 15000,
  "final_row_count": 15000,
  "weak_topic_rows": 7500,
  "weak_topic_fraction": 0.5,
  "trace_chars": {
    "mean": 2513.9,
    "median": 2399.0,
    "p95": 3387.0
  },
  "template_tokens": {
    "mean": 1107.0,
    "median": 1070.0,
    "p95": 1600.0
  },
  "topic_counts": {
    "olympiads": 5653,
    "geometry": 4901,
    "sequence_recurrence": 155

In [6]:
# Spot-check supplements (run after RUN_V2_SUPPLEMENTS)
SPOT_LONG = 5
SPOT_MULTI = 5
for label, path in [
    ("long-trace", SFT_SOURCES_DIR / "numina_long_trace.jsonl"),
    ("multi-blank", SFT_SOURCES_DIR / "numina_multi_blank_synth.jsonl"),
]:
    if not path.is_file():
        print(f"Skip {label}: missing {path.name}")
        continue
    rows = read_jsonl(path)
    k = SPOT_LONG if label == "long-trace" else SPOT_MULTI
    for i, row in enumerate(random.sample(rows, k=min(k, len(rows))), 1):
        print("=" * 72)
        print(
            f"[{label} {i}] {row.get('source_id')} | trace_chars={row['trace_chars']} | "
            f"blanks={row.get('blank_count', row['question'].count('[ANS]'))} | mode={row.get('supplement_mode')}"
        )
        print("Q:", row["question"][:280].replace("\n", " "), "…")
        print("Final:", last_non_empty_line(row["response"]))

[long-trace 1] olympiads:584019 | trace_chars=5352 | blanks=0 | mode=long_trace
Q: Do either (1) or (2):  (1) Prove that \[ \int_1^k \lfloor x \rfloor f '(x) \, dx = \lfloor k \rfloor f(k) - \sum_{n=1}^{\lfloor k \rfloor} f(n) \] where \( k > 1 \), and \( \lfloor z \rfloor \) denotes the greatest integer ≤ \( z \). Find a similar expression for: \[ \int_1^k \lf …
Final: \boxed{1.26 \text{s}}
[long-trace 2] olympiads:298374 | trace_chars=4190 | blanks=0 | mode=long_trace
Q: A set of positive integers is called fragrant if it has at least 2 elements, and for every element in the set, there is at least one other element in the set such that the two elements have a common prime divisor. Let \( P(n) = n^2 + n + 1 \). What is the smallest positive intege …
Final: \boxed{6}
[long-trace 3] olympiads:847085 | trace_chars=4889 | blanks=0 | mode=long_trace
Q: In rhombus \(ABCD\), the side length is 1, and \(\angle ABC = 120^\circ\). Let \(E\) be any point on the extension of \(BC\). If \(AE\) int

### 8.1 Spot-check `numina_long_trace` rows with `trace_chars ≥ 6000`

Only ~51 of the 1,500 long-trace supplement rows hit the strict ≥6000 wrapped threshold. Sample 10 for manual review (seed 42).

In [9]:
NUMINA_LONG_TRACE_PATH = SFT_SOURCES_DIR / "numina_long_trace.jsonl"
SPOT_LONG_GE6000_N = 10
SPOT_LONG_GE6000_SEED = 42

if not NUMINA_LONG_TRACE_PATH.is_file():
    print(f"Missing {NUMINA_LONG_TRACE_PATH} — run long-trace builder first.")
else:
    long_trace_rows = read_jsonl(NUMINA_LONG_TRACE_PATH)
    ge6000 = [r for r in long_trace_rows if r.get("trace_chars", 0) >= 6000]
    print(
        f"numina_long_trace.jsonl: {len(long_trace_rows):,} rows, "
        f"{len(ge6000):,} with trace_chars >= 6000"
    )
    if not ge6000:
        print("No rows >= 6000 to spot-check.")
    else:
        rng = random.Random(SPOT_LONG_GE6000_SEED)
        spot_ge6000 = rng.sample(ge6000, k=min(SPOT_LONG_GE6000_N, len(ge6000)))
        for i, row in enumerate(spot_ge6000, 1):
            print("=" * 72)
            print(
                f"[{i}] {row.get('source_id')} | hf={row.get('hf_source')} | topic={row.get('topic')} | "
                f"weak={row.get('weak_topic')} | tokens={row.get('template_tokens')} | "
                f"chars={row['trace_chars']} | mode={row.get('supplement_mode')}"
            )
            print("Q:", row["question"][:320].replace("\n", " "), "…")
            print("A:", str(row.get("answer", ""))[:120])
            print("Final line:", last_non_empty_line(row["response"]))
            print("Thinking wrapper:", has_thinking_wrapper(row["response"]))
            print("Tail:", row["response"][-280:].replace("\n", " "))

numina_long_trace.jsonl: 1,500 rows, 51 with trace_chars >= 6000
[1] cn_k12:434587 | hf=cn_k12 | topic=cn_k12 | weak=False | tokens=2365 | chars=6589 | mode=long_trace
Q: Given that the distance from the focus F of a parabola to its directrix is 4, if the distance from a point P on the parabola to the y-axis is 1, then the distance from point P to the focus F is equal to A) 2 B) 3 C) 4 D) 5 …
A: PF = 5
Final line: \boxed{PF = 5}
Thinking wrapper: True
Tail: to -1. The distance from P to the directrix is: $$ PD = |p - (-4)| = |p + 4| $$ But since P is on the left side of the y-axis, we have: $$ PD = -(-1) + 4 = 5 $$ Then the distance PF equals PD since P lies on the parabola: $$ PF = 5 $$  As such we find: $$ </think>  \boxed{PF = 5}
[2] olympiads:85323 | hf=olympiads | topic=olympiads | weak=False | tokens=3006 | chars=6315 | mode=long_trace
Q: 1) \( y = -\frac{\cos x}{\sin^2 x} + \ln \tan \frac{x}{2}; \; y' \) ?  2) \( y = \arcsin (\cos x); \; y' \) ?  3) \( r = \varphi^2 \arccos \fra

### 8.2 Spot-check `numina_multi_blank_synth` (judger + training prompt)

All rows should have **2–4** `[ANS]` placeholders, a comma-separated multi-`\boxed{}` final line, and `blank_count` matching `extract_all_boxed` on the final (pub-002 / `judger.extract_all_boxed`). Sample 10 composed rows (seed 42) and print the `build_prompt_multi_blank` user tail used at SFT/inference.

In [10]:
from scripts.sft_prompt import (
    build_prompt_multi_blank,
    extract_all_boxed,
    n_ans_placeholders,
)

NUMINA_MULTI_BLANK_PATH = SFT_SOURCES_DIR / "numina_multi_blank_synth.jsonl"
SPOT_MULTI_BLANK_N = 10
SPOT_MULTI_BLANK_SEED = 42


def _audit_multi_blank(rows: list[dict]) -> dict[str, int]:
    issues: Counter[str] = Counter()
    for row in rows:
        q = row["question"]
        bc = row.get("blank_count", n_ans_placeholders(q))
        ans_n = n_ans_placeholders(q)
        if bc != ans_n:
            issues["blank_count_vs_[ANS]"] += 1
        if bc < 2:
            issues["too_few_blanks"] += 1
        if bc > 4:
            issues["too_many_blanks"] += 1
        final = last_non_empty_line(row["response"])
        boxed = extract_all_boxed(final)
        if len(boxed) != bc:
            issues["boxed_count_vs_blank_count"] += 1
        if not row.get("options") and bc <= 1:
            issues["single_blank_freeform"] += 1
    return dict(issues)


if not NUMINA_MULTI_BLANK_PATH.is_file():
    print(f"Missing {NUMINA_MULTI_BLANK_PATH} — run `build_sft_corpus.py multi-blank` first.")
else:
    mb_rows = read_jsonl(NUMINA_MULTI_BLANK_PATH)
    blank_hist = Counter(r.get("blank_count", n_ans_placeholders(r["question"])) for r in mb_rows)
    mode_hist = Counter(r.get("supplement_mode", "?") for r in mb_rows)
    print(
        f"numina_multi_blank_synth.jsonl: {len(mb_rows):,} rows | "
        f"blank_count {dict(sorted(blank_hist.items()))} | modes {dict(mode_hist)}"
    )
    audit = _audit_multi_blank(mb_rows)
    if audit:
        print("AUDIT ISSUES:", audit)
    else:
        print("Audit OK: blank_count, [ANS], and final-line \\boxed{} groups align.")

    rng = random.Random(SPOT_MULTI_BLANK_SEED)
    spot_mb = rng.sample(mb_rows, k=min(SPOT_MULTI_BLANK_N, len(mb_rows)))
    for i, row in enumerate(spot_mb, 1):
        q = row["question"]
        bc = row.get("blank_count", n_ans_placeholders(q))
        final = last_non_empty_line(row["response"])
        boxed = extract_all_boxed(final)
        _sys, user = build_prompt_multi_blank(q, row.get("options"))
        print("=" * 72)
        print(
            f"[{i}] {row.get('source_id')} | blanks={bc} | mode={row.get('supplement_mode')} | "
            f"trace_chars={row.get('trace_chars')} | composed_from={row.get('composed_from', [])[:3]}"
        )
        print("Q:", q[:320].replace("\n", " "), "…")
        print("Final:", final[:200], ("…" if len(final) > 200 else ""))
        print("extract_all_boxed:", boxed)
        print("answer field:", str(row.get("answer", ""))[:120])
        print("Prompt user (tail):", user[-240:].replace("\n", " "))

numina_multi_blank_synth.jsonl: 1,500 rows | blank_count {2: 719, 3: 463, 4: 318} | modes {'composed': 1500}
Audit OK: blank_count, [ANS], and final-line \boxed{} groups align.
[1] cn_k12:207415+cn_k12:394244 | blanks=2 | mode=composed | trace_chars=4488 | composed_from=['cn_k12:207415', 'cn_k12:394244']
Q: (a) Given an ellipse $E$: $\frac{x^{2}}{a^{2}}+\frac{y^{2}}{b^{2}}=1 (a > b > 0)$ with one of its foci being $F_2(1,0)$, and the ellipse passes through a fixed point $M(1, \frac{\sqrt{2}}{2})$. (I) Find the standard equation of ellipse $E$; (II) Let point $Q(2,0)$, draw a line $l$ passing through point $F_2$ and inters …
Final: \boxed{2}, \boxed{\frac{3}{4}-\ln 2} 
extract_all_boxed: ['2', '\\frac{3}{4}-\\ln 2']
answer field: 2, \frac{3}{4}-\ln 2
Prompt user (tail): $, and $x_{1}\in(0,\frac{1}{2})$. If $h(x_{1})-h(x_{2})>m$ holds, find the maximum value of $m$.  Answer: [ANS]  The problem has 2 [ANS] blanks. After your reasoning, give 2 comma-separated \boxed{} values in order: \box

### 9.2 OpenR1 config & helpers

Set `RUN_OPENR1_1K = True` to download from HuggingFace and build the 1k probe corpus. Requires `HF_TOKEN` in Colab secrets for reliable Hub access.

In [7]:
import statistics
import subprocess
from datetime import datetime, timezone

from datasets import load_dataset
from scripts.sft_prompt import is_figure_dependent, split_thinking_response

OPENR1_SOURCE = "openr1_math"
OPENR1_DATASET_ID = "open-r1/OpenR1-Math-220k"
OPENR1_CONFIG = "default"
OPENR1_MAX_RESPONSE_CHARS = 10_000
OPENR1_MIN_RESPONSE_CHARS = 2_000
OPENR1_SAMPLE_SIZE = 1_000
OPENR1_SAMPLE_SEED = 42
DECONTAM_PREFIX_CHARS = 200

OPENR1_READY_PATH = SFT_SOURCES_DIR / f"{OPENR1_SOURCE}_ready.jsonl"
OPENR1_STATS_PATH = SFT_SOURCES_DIR / f"{OPENR1_SOURCE}_stats.json"
OPENR1_REJECTS_PATH = SFT_SOURCES_DIR / f"{OPENR1_SOURCE}_rejects.jsonl"
OPENR1_CORPUS_PATH = REPO_ROOT / "data" / "sft_corpus_openr1_1k.jsonl"
OPENR1_MANIFEST_PATH = REPO_ROOT / "data" / "sft_corpus_openr1_1k_manifest.json"

RUN_OPENR1_1K = True  # set True to download + build


def _git_head_short() -> Optional[str]:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
        ).strip()[:12]
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def load_decontam_prefixes(*paths: Path, prefix_chars: int = DECONTAM_PREFIX_CHARS) -> list[str]:
    prefixes: list[str] = []
    for path in paths:
        if not path.is_file():
            continue
        with open(path) as f:
            for line in f:
                norm = normalize_question_for_overlap(json.loads(line)["question"][:prefix_chars])
                if len(norm) >= 20:
                    prefixes.append(norm)
    return prefixes


def decontam_hit(problem: str, prefixes: list[str]) -> bool:
    p_full = normalize_question_for_overlap(problem)
    p_prefix = normalize_question_for_overlap(problem[:DECONTAM_PREFIX_CHARS])
    if p_full in competition_question_keys:
        return True
    for pref in prefixes:
        if pref in p_full or p_prefix in pref or pref in p_prefix:
            return True
    return False


def pick_correct_generation(ex: dict[str, Any]) -> Optional[str]:
    gens = ex.get("generations") or []
    flags = ex.get("correctness_math_verify") or []
    for gen, ok in zip(gens, flags):
        if ok and gen and len(gen) <= OPENR1_MAX_RESPONSE_CHARS:
            return str(gen).strip()
    return None


def assemble_thinking_response(reasoning: str, final_line: str) -> str:
    parts = [THINKING_OPEN]
    if reasoning.strip():
        parts.append(reasoning.strip())
    parts.append(THINKING_CLOSE)
    parts.append("")
    parts.append(final_line.strip())
    return "\n".join(parts)


def format_openr1_response(raw_trace: str) -> tuple[Optional[str], Optional[str]]:
    trace = raw_trace.strip()
    if not trace:
        return None, None

    has_wrapper = THINKING_OPEN in trace and THINKING_CLOSE in trace
    if not has_wrapper:
        norm, answer = normalize_freeform_response(trace)
        if norm is None or answer is None:
            return None, None
        return wrap_thinking_response(norm), answer.strip()

    reasoning, after_close = split_thinking_response(trace)
    after_close = after_close.strip()
    boxed_after = extract_boxed_answer(after_close) if after_close else None
    boxed_inside = extract_boxed_answer(reasoning) if reasoning else None

    if boxed_after is not None:
        final_line = last_non_empty_line(after_close)
        if not FREEFORM_FINAL_RE.match(final_line):
            boxed = last_boxed_only_string(after_close)
            if boxed is None:
                return None, None
            final_line = boxed.strip()
        response = assemble_thinking_response(reasoning, final_line)
        answer = extract_boxed_answer(final_line)
        return (response, answer.strip()) if answer else (None, None)

    if boxed_inside is not None:
        boxed = last_boxed_only_string(reasoning)
        if boxed is None:
            return None, None
        idx = reasoning.rfind(boxed)
        reasoning_clean = reasoning[:idx].rstrip() if idx >= 0 else reasoning
        final_line = boxed.strip()
        response = assemble_thinking_response(reasoning_clean, final_line)
        return response, extract_boxed_answer(final_line)

    return None, None


def dist_stats(values: list[int | float]) -> dict[str, float]:
    if not values:
        return {"p25": 0.0, "p50": 0.0, "p95": 0.0, "mean": 0.0, "median": 0.0}
    s = sorted(values)

    def pct(p: float) -> float:
        idx = max(0, int(p * len(s)) - 1)
        return float(s[idx])

    return {
        "p25": round(pct(0.25), 1),
        "p50": round(pct(0.50), 1),
        "p95": round(pct(0.95), 1),
        "mean": round(statistics.mean(s), 1),
        "median": round(statistics.median(s), 1),
    }


def qualify_openr1_row(
    ex: dict[str, Any],
    *,
    idx: int,
    decontam_prefixes: list[str],
) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    reject_base = {"source": OPENR1_SOURCE, "source_id": f"openr1:{idx}"}
    problem = (ex.get("problem") or "").strip()
    if not problem:
        return None, {**reject_base, "reason": "empty_problem"}

    trace = pick_correct_generation(ex)
    if trace is None:
        return None, {**reject_base, "reason": "no_correct_gen"}

    if contains_cjk(problem) or contains_cjk(trace):
        return None, {**reject_base, "reason": "cjk_text", "question_preview": problem[:200]}
    if is_figure_dependent(problem) or is_figure_dependent(trace):
        return None, {**reject_base, "reason": "figure_dependent", "question_preview": problem[:200]}
    if decontam_hit(problem, decontam_prefixes):
        return None, {
            **reject_base,
            "reason": "decontam",
            "question_preview": problem[:200],
        }

    response, answer = format_openr1_response(trace)
    if response is None or answer is None or not str(answer).strip():
        return None, {**reject_base, "reason": "bad_format", "trace_preview": trace[:200]}
    if not has_thinking_wrapper(response):
        return None, {**reject_base, "reason": "missing_thinking_wrapper"}
    if len(response) < OPENR1_MIN_RESPONSE_CHARS:
        return None, {**reject_base, "reason": "short_response", "trace_chars": len(response)}
    if len(response) > OPENR1_MAX_RESPONSE_CHARS:
        return None, {**reject_base, "reason": "long_response", "trace_chars": len(response)}
    if not validate_final_line(response, "freeform"):
        return None, {**reject_base, "reason": "bad_final_line", "response_tail": response[-120:]}

    messages = render_training_messages(problem, None, response)
    template_tokens = count_template_tokens(tokenizer, messages)
    if template_tokens > MAX_TEMPLATE_TOKENS:
        return None, {
            **reject_base,
            "reason": "too_long_tokens",
            "template_tokens": template_tokens,
        }

    ready = {
        "source": OPENR1_SOURCE,
        "source_id": f"openr1:{idx}",
        "task_type": "freeform",
        "question": problem,
        "options": None,
        "response": response,
        "answer": answer.strip(),
        "topic": "openr1",
        "weak_topic": False,
        "thinking_template": THINKING_TEMPLATE,
        "trace_chars": len(response),
        "template_tokens": template_tokens,
    }
    return ready, None


def run_openr1_1k_build() -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    decontam_prefixes = load_decontam_prefixes(PUBLIC_PATH, PRIVATE_PATH)
    print(f"Decontam prefixes (first {DECONTAM_PREFIX_CHARS} chars): {len(decontam_prefixes)}")

    ds = load_dataset(OPENR1_DATASET_ID, OPENR1_CONFIG, split="train")
    print(f"Loaded {OPENR1_DATASET_ID} [{OPENR1_CONFIG}]: {len(ds):,} rows")
    print("Columns:", ds.column_names)

    ready: list[dict[str, Any]] = []
    rejects: list[dict[str, Any]] = []
    reject_counts: Counter = Counter()

    for idx, ex in enumerate(tqdm(ds, desc="OpenR1 filter")):
        row, reject = qualify_openr1_row(ex, idx=idx, decontam_prefixes=decontam_prefixes)
        if row is not None:
            ready.append(row)
        else:
            assert reject is not None
            rejects.append(reject)
            reject_counts[reject["reason"]] += 1

    print(f"After filters: {len(ready):,} ready / {len(ds):,} input ({100 * len(ready) / len(ds):.1f}%)")
    print("Reject counts:", dict(reject_counts))

    rng = random.Random(OPENR1_SAMPLE_SEED)
    rng.shuffle(ready)
    if len(ready) < OPENR1_SAMPLE_SIZE:
        raise RuntimeError(
            f"Only {len(ready)} rows pass filters; need {OPENR1_SAMPLE_SIZE} for sft-002a probe."
        )
    corpus_rows = ready[:OPENR1_SAMPLE_SIZE]

    validate_ready_corpus(corpus_rows, OPENR1_SOURCE, require_thinking_wrapper=True)

    write_jsonl(OPENR1_READY_PATH, ready)
    write_jsonl(OPENR1_REJECTS_PATH, rejects)
    write_jsonl(OPENR1_CORPUS_PATH, corpus_rows)

    trace_chars = [r["trace_chars"] for r in corpus_rows]
    template_tokens = [r["template_tokens"] for r in corpus_rows]
    manifest = {
        "corpus_id": "sft-002a",
        "thinking_template": THINKING_TEMPLATE,
        "source_dataset": OPENR1_DATASET_ID,
        "source_config": OPENR1_CONFIG,
        "sample_seed": OPENR1_SAMPLE_SEED,
        "target_size": OPENR1_SAMPLE_SIZE,
        "final_row_count": len(corpus_rows),
        "input_rows": len(ds),
        "after_filter_rows": len(ready),
        "filter_retention_rate": round(len(ready) / len(ds), 4),
        "filter_drops": dict(reject_counts),
        "decontam_hits": reject_counts.get("decontam", 0),
        "max_response_chars": OPENR1_MAX_RESPONSE_CHARS,
        "min_response_chars": OPENR1_MIN_RESPONSE_CHARS,
        "trace_chars": dist_stats(trace_chars),
        "template_tokens": dist_stats(template_tokens),
        "corpus_path": str(OPENR1_CORPUS_PATH.relative_to(REPO_ROOT)),
        "corpus_sha256": file_sha256(OPENR1_CORPUS_PATH),
        "ready_pool_path": str(OPENR1_READY_PATH.relative_to(REPO_ROOT)),
        "ready_pool_sha256": file_sha256(OPENR1_READY_PATH),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "git_commit": _git_head_short(),
    }
    OPENR1_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")

    stats = {
        "source": OPENR1_SOURCE,
        "input_rows": len(ds),
        "ready_rows": len(ready),
        "corpus_rows": len(corpus_rows),
        "reject_counts": dict(reject_counts),
        "manifest_path": str(OPENR1_MANIFEST_PATH),
    }
    OPENR1_STATS_PATH.write_text(json.dumps(stats, indent=2) + "\n")

    print(f"Wrote {OPENR1_CORPUS_PATH} ({len(corpus_rows):,} rows)")
    print(f"Wrote {OPENR1_MANIFEST_PATH}")
    print(json.dumps(manifest, indent=2))
    return corpus_rows, rejects, manifest

In [8]:
if RUN_OPENR1_1K:
    openr1_corpus, openr1_rejects, openr1_manifest = run_openr1_1k_build()
else:
    print("Skip OpenR1 build (RUN_OPENR1_1K=False).")
    if OPENR1_CORPUS_PATH.is_file():
        openr1_corpus = read_jsonl(OPENR1_CORPUS_PATH)
        print(f"Loaded existing corpus: {len(openr1_corpus):,} rows from {OPENR1_CORPUS_PATH.name}")
    else:
        openr1_corpus = []
        print(f"No corpus at {OPENR1_CORPUS_PATH} — set RUN_OPENR1_1K=True to build.")

Decontam prefixes (first 200 chars): 2069


Generating train split: 100%|██████████| 93733/93733 [00:07<00:00, 12932.22 examples/s]


Loaded open-r1/OpenR1-Math-220k [default]: 93,733 rows
Columns: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages']


OpenR1 filter: 100%|██████████| 93733/93733 [01:53<00:00, 825.07it/s] 


After filters: 26,684 ready / 93,733 input (28.5%)
Reject counts: {'no_correct_gen': 66106, 'figure_dependent': 211, 'cjk_text': 450, 'short_response': 272, 'bad_format': 6, 'bad_final_line': 4}
Validation passed for 1000 ready rows (openr1_math).
Wrote /home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/data/sft_corpus_openr1_1k.jsonl (1,000 rows)
Wrote /home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/data/sft_corpus_openr1_1k_manifest.json
{
  "corpus_id": "sft-002a",
  "thinking_template": "explicit_redacted_thinking",
  "source_dataset": "open-r1/OpenR1-Math-220k",
  "source_config": "default",
  "sample_seed": 42,
  "target_size": 1000,
  "final_row_count": 1000,
  "input_rows": 93733,
  "after_filter_rows": 26684,
  "filter_retention_rate": 0.2847,
  "filter_drops": {
    "no_correct_gen": 66106,
    "figure_dependent": 211,
    "cjk_text": 450,
    "short_response": 272,
    "bad_format": 6,
    "bad_final_line": 4
  },
  

### 9.3 Spot-check 10 OpenR1 rows

Manual review checklist: long exploratory R1 trace (not Numina-template stub); `<think>...</think>` wraps reasoning; `\boxed{}` after closing tag; no CJK; no figure-dependent language.

In [9]:
SPOT_OPENR1 = 10
if not openr1_corpus:
    print("No OpenR1 corpus loaded — run §9.2 with RUN_OPENR1_1K=True first.")
else:
    spot_openr1 = random.sample(openr1_corpus, k=min(SPOT_OPENR1, len(openr1_corpus)))
    spot_failures = 0
    for i, row in enumerate(spot_openr1, 1):
        resp = row["response"]
        checks = {
            "thinking_wrapper": has_thinking_wrapper(resp),
            "boxed_after_close": THINKING_CLOSE in resp
            and extract_boxed_answer(resp.split(THINKING_CLOSE, 1)[1]) is not None,
            "no_cjk": not contains_cjk(row["question"]) and not contains_cjk(resp),
            "no_figure": not is_figure_dependent(row["question"]),
            "long_enough": row["trace_chars"] >= OPENR1_MIN_RESPONSE_CHARS,
        }
        failed = [k for k, ok in checks.items() if not ok]
        if failed:
            spot_failures += 1
        print("=" * 72)
        print(
            f"[{i}] {row['source_id']} | chars={row['trace_chars']} | "
            f"tokens={row['template_tokens']} | checks_ok={not failed}"
        )
        if failed:
            print("FAILED:", failed)
        print("Q:", row["question"][:280].replace("\n", " "), "…")
        print("Final:", last_non_empty_line(resp))
        print("Tail:", resp[-420:].replace("\n", " "))
    print("=" * 72)
    print(f"Spot check: {len(spot_openr1) - spot_failures}/{len(spot_openr1)} passed automated checks.")
    if spot_failures > 1:
        print("WARNING: >1 row failed — tighten filters and rebuild before training.")

[1] openr1:92842 | chars=3225 | tokens=1005 | checks_ok=True
Q: ## Task 11/77  What is the probability that the second roll of a fair die will result in a higher number than the first? …
Final: \boxed{\dfrac{5}{12}}
Tail: 2.  Alternatively, if I didn't think of that symmetry argument, I could have just stuck with my original counting method and still arrived at the same answer, which is reassuring.  So, conclusion: There are 15 favorable outcomes where the second roll is higher, out of 36 total. That reduces to 5/12. So the probability is 5/12.  **Final Answer** The probability is \boxed{\dfrac{5}{12}}. </think>  \boxed{\dfrac{5}{12}}
[2] openr1:40964 | chars=4866 | tokens=2390 | checks_ok=True
Q: 14. A teacher assigned the task of drawing a triangle $A B C$ at will on the notebook, drawing the three altitudes $A A^{\prime}, B B^{\prime}, C C^{\prime}$ and measuring their respective lengths. Five students found the measurements reported in the answers below. Which of them  …
Final: \bo

## 10. Optional: copy artifacts to Drive

Keeps `sft_sources/` recoverable across Colab disconnects.


In [ ]:
import shutil

if DRIVE_PROJECT_ROOT is not None:
    drive_data = DRIVE_PROJECT_ROOT / "data"
    drive_sources = drive_data / "sft_sources"
    drive_sources.mkdir(parents=True, exist_ok=True)
    paths = [
        MATH_READY_PATH,
        MATH_STATS_PATH,
        MATH_REJECTS_PATH,
        NUMINA_READY_PATH,
        NUMINA_STATS_PATH,
        NUMINA_REJECTS_PATH,
        AGIEVAL_MCQ_READY_PATH,
        AGIEVAL_MCQ_STATS_PATH,
        AGIEVAL_MCQ_REJECTS_PATH,
        REPO_ROOT / "data" / "sft_corpus.jsonl",
        REPO_ROOT / "data" / "sft_corpus_manifest.json",
        REPO_ROOT / "data" / "sft_corpus_v2.jsonl",
        REPO_ROOT / "data" / "sft_corpus_v2_manifest.json",
        OPENR1_CORPUS_PATH,
        OPENR1_MANIFEST_PATH,
        OPENR1_READY_PATH,
        OPENR1_STATS_PATH,
        OPENR1_REJECTS_PATH,
        SFT_SOURCES_DIR / "numina_long_trace.jsonl",
        SFT_SOURCES_DIR / "numina_multi_blank_synth.jsonl",
    ]
    for path in paths:
        if not path.is_file():
            continue
        dest = (
            drive_data / path.name
            if path.name.startswith("sft_corpus")
            else drive_sources / path.name
        )
        shutil.copy2(path, dest)
    print(f"Copied SFT artifacts under {drive_data}")
else:
    print("Skip: no Drive mount.")